# Network construction

This notebook builds and verifies the drug–target–protein network.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../data/README.md) and
[DataPreparation.ipynb](../../data/DataPreparation.ipynb), you can build the
network using the functions in [`src/dtpkg/network/`](../../src/dtpkg/network/).


## Paths and run options

Replace `...` with quoted paths. Use the same `DATA_DIR` as the data preparation notebook;
`GRAPH_PATH` is the output network. Enable `RUN_BUILD` to build and verify the
network, or `RUN_VERIFY` to check an existing graph.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = ...  # DTP-KG checkout folder
DATA_DIR = ...  # Local data folder; see data/README.md
GRAPH_PATH = ...  # Output GraphML file

if any(path is ... for path in (PROJECT_ROOT, DATA_DIR, GRAPH_PATH)):
    raise ValueError("Replace ... in PROJECT_ROOT, DATA_DIR, GRAPH_PATH with your local paths.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DATA_DIR = Path(DATA_DIR).expanduser().resolve()
GRAPH_PATH = Path(GRAPH_PATH).expanduser().resolve()

RUN_BUILD = False
RUN_VERIFY = False

## Check the prepared inputs

The data preparation notebook prepares the interaction tables and resolved labels. This task reads
those files without repeating source extraction.


In [ ]:
if RUN_BUILD:
    required = [DATA_DIR / "interactions" / name for name in (
        "DDI_positive_pairs.csv", "DPI_enriched.csv", "PPI_enriched.csv")]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Run data/DataPreparation.ipynb first; missing: " + ", ".join(missing))


## Build the network

The constructor reads `DDI_positive_pairs.csv`, `DPI_enriched.csv`, and
`PPI_enriched.csv` from `DATA_DIR / "interactions"`. The resulting graph is simple,
undirected, and unweighted. Targets take precedence over other proteins; PPI
self-loops, duplicate edges, and isolated nodes are removed. No embeddings are used.

In [ ]:
if RUN_BUILD:
    from dtpkg.network.construct_network import main as construct_network

    construct_network(["--data-dir", str(DATA_DIR), "--out", str(GRAPH_PATH)])

## Verify the graph

Verification compares graph DDI edges with the positive pair table and reports
aggregate counts and a content hash. A mismatch raises an error. The study graph
has 24,457 nodes and 1,714,088 edges; different source releases can change these counts.

| Node type | Study count |
|---|---:|
| Drug | 5,325 |
| Target | 2,952 |
| Other protein | 16,180 |

In [ ]:
if RUN_BUILD or RUN_VERIFY:
    from dtpkg.cli.verify_graph_source import main as verify_graph_source

    verify_graph_source(["--data-dir", str(DATA_DIR), "--graph", str(GRAPH_PATH)])